<a href="https://colab.research.google.com/github/barkain/recsys-2026/blob/r54-second-gen-supervised-retriever/r54-second-gen-supervised-retriever-notebook" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R54 Phase 3 Full — Colab T4 Runner

Trains R54 Phase 3 folds 1-4 on Colab GPU. Bring back per-fold `oof_lists.json` artifacts and run integration locally.

**Plan:** fold 1 first as a sanity check, then folds 2-4 if fold 1 looks good. Fold 0 is reused from the local Phase 3 smoke run.

**Reference:** `docs/r54_colab_runbook.md`

## Cell 1: GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/barkain/recsys-2026.git
%cd recsys-2026
!git checkout r54-second-gen-supervised-retriever
!git log --oneline -3

## Cell 2: Install deps via uv

In [ ]:
!pip install -q uv
!uv sync
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Cell 3: Upload R12 payload (one-time)

Upload your local `exp/eval/_R12_all_turns_payload.pkl` (~114MB).

Either drag-drop in the Colab files panel, or use the upload widget below:

In [ ]:
import os
os.makedirs("exp/eval", exist_ok=True)

from google.colab import files
print("Upload _R12_all_turns_payload.pkl now")
uploaded = files.upload()
import shutil
src = next(iter(uploaded.keys()))
shutil.move(src, "exp/eval/_R12_all_turns_payload.pkl")
print("Placed at:", os.path.exists("exp/eval/_R12_all_turns_payload.pkl"),
      "size:", os.path.getsize("exp/eval/_R12_all_turns_payload.pkl"))

**Option B (alternative):** mount Drive and copy from there.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/r54/_R12_all_turns_payload.pkl exp/eval/
```

## Cell 4: Pre-download HF datasets (one-time, ~2-3 min)

In [ ]:
from datasets import load_dataset
_ = load_dataset("talkpl-ai/TalkPlayData-Challenge-Track-Metadata")
_ = load_dataset("talkpl-ai/TalkPlayData-Challenge-Dataset")
print("HF datasets cached.")

## Cell 5: SANITY — fold 1 only

**Stop here and inspect output before continuing.** Expect:
- `Device: cuda`
- `fold 1: train_dev=6400  train_split=20000  total=26400`
- batches logging every 50, much faster than CPU (~0.3-1s/batch on T4)
- Final line: `fold 1 val hit@200: XXX/1600 (≥ 0.539)` — should be at or above Phase 2 fold-1 baseline.

In [ ]:
!uv run python scripts/expR54_phase3_full5fold_train.py --fold 1 --device cuda --no-aggregate

## Cell 6: Download fold 1 artifact (do this before running folds 2-4)

In [ ]:
import os, shutil
os.makedirs("/content/r54_phase3_artifacts", exist_ok=True)
shutil.copy("cache/r54/phase3_full/fold_1/oof_lists.json",
            "/content/r54_phase3_artifacts/fold_1_oof_lists.json")
!cd /content && zip -r r54_phase3_fold1.zip r54_phase3_artifacts/
from google.colab import files
files.download("/content/r54_phase3_fold1.zip")

## Cell 7: Folds 2, 3, 4 (only after fold 1 is verified)

In [ ]:
for fold_i in [2, 3, 4]:
    print(f"\n=== Running fold {fold_i} ===")
    !uv run python scripts/expR54_phase3_full5fold_train.py --fold {fold_i} --device cuda --no-aggregate

## Cell 8: Download all artifacts

Only `oof_lists.json` per fold is needed locally (28MB each × 4 = ~112MB total).

In [ ]:
import os, shutil
os.makedirs("/content/r54_phase3_artifacts", exist_ok=True)
for fold_i in [1, 2, 3, 4]:
    src = f"cache/r54/phase3_full/fold_{fold_i}/oof_lists.json"
    if os.path.exists(src):
        shutil.copy(src, f"/content/r54_phase3_artifacts/fold_{fold_i}_oof_lists.json")
        print(f"  fold {fold_i}: copied")
    else:
        print(f"  fold {fold_i}: MISSING")
!ls -la /content/r54_phase3_artifacts/
!cd /content && zip -r r54_phase3_artifacts.zip r54_phase3_artifacts/
from google.colab import files
files.download("/content/r54_phase3_artifacts.zip")

## Drive backup (optional)

Colab storage is temporary. If the session disconnects before you download, the artifacts are lost.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/r54_phase3_artifacts.zip /content/drive/MyDrive/r54/
```

## Local: install artifacts and evaluate

Run these locally after downloading:

```bash
cd /Users/nadavbarkai/dev/recsys-2026
unzip ~/Downloads/r54_phase3_artifacts.zip -d /tmp/r54_artifacts
for f in 1 2 3 4; do
  mkdir -p cache/r54/phase3_full/fold_$f
  cp /tmp/r54_artifacts/r54_phase3_artifacts/fold_${f}_oof_lists.json \
     cache/r54/phase3_full/fold_$f/oof_lists.json
done
mkdir -p cache/r54/phase3_full/fold_0
cp cache/r54/phase3_smoke/fold_0/oof_lists.json cache/r54/phase3_full/fold_0/oof_lists.json

# Aggregate into single oof_r54_lists.json
uv run python -c "
import sys; sys.path.insert(0, '.')
from scripts.expR54_phase3_full5fold_train import aggregate_oof_lists
from pathlib import Path
aggregate_oof_lists(Path('cache/r54/phase3_full'), 8000)
"

# Evaluate
uv run python scripts/expR54_phase3_full5fold_standalone.py
uv run python scripts/expR54_phase3_full5fold_integration.py
```